# Линейный и нелинейный фильтры одной модели с переключениями

Используем тот же конфиг `toy_continuous_additive`: два состояния цепи, двумерная треугольная метка $Y$, постоянное СКО наблюдений $0.7$,
$$dX_t^k=Y_t^k\,dt+0.7\,dW_t^k.$$
Для каждого из трёх семян скрытая траектория и наблюдение общие для всех фильтров. Шаг генерации $0.001$, фильтрации $0.02$ и $0.01$; горизонт $T=4$.

Линейный фильтр строится по расширению состояния из [статьи Борисова и Куринова](../article/intels.pdf), формулы (2)–(4), (10)–(11), страницы 4–5 и 8. Это точное линейное мартингальное представление скачкообразной модели, без разложения Тейлора:
$$V_t=\operatorname{col}(\theta_t,Y_t^1\theta_t,Y_t^2\theta_t)\in\mathbb R^6,
\qquad dV_t=A V_t\,dt+dM_t,\qquad dX_t=H V_t\,dt+R^{1/2}dW_t.$$
Здесь $R=0.7^2 I$ обозначает **ковариацию** шума, а не его СКО.

Нелинейный фильтр оценивает условную плотность. Калман–Бьюси даёт наилучшую **линейную** оценку по наблюдениям в этой негауссовской системе. Его компоненты $\theta$ могут выходить за $[0,1]$; мы их не обрезаем и не интерпретируем как апостериорные вероятности.

## Матрицы расширенной системы

Порядок компонент: $(\theta_0,\theta_1,Y^1\theta_0,Y^1\theta_1,Y^2\theta_0,Y^2\theta_1)$. Пусть $Q$ — строчный генератор, $Q_\circ=Q-\operatorname{diag}(Q_{nn})$, $\mu_k(n)=E[Y^k\mid\theta=n]$. Тогда
$$A=\begin{pmatrix}
Q^\top&0&0\\
\operatorname{diag}(\mu_1)Q_\circ^\top&\operatorname{diag}(Q_{nn})&0\\
\operatorname{diag}(\mu_2)Q_\circ^\top&0&\operatorname{diag}(Q_{nn})
\end{pmatrix}.$$
Матрица $H$ суммирует компоненты блоков $Y^1\theta$ и $Y^2\theta$.

Начальное распределение стационарно. Для $S=E[V_0V_0^\top]$ ожидаемая скорость квадратичной характеристики мартингала из (4) равна $B=-(AS+SA^\top)$. Используем именно **безусловную** матрицу $B$. Все моменты сброса вычисляем той же пространственной квадратурой, что и нелинейный фильтр; конечная сетка остаётся источником погрешности относительно исходного непрерывного треугольного закона.

При $K_0=S-m_0m_0^\top$ уравнения Калмана–Бьюси:
$$G_t=K_tH^\top R^{-1},\qquad
dm_t=Am_t\,dt+G_t(dX_t-Hm_t\,dt),$$
$$\dot K_t=AK_t+K_tA^\top+B-G_tRG_t^\top.$$
Численный шаг использует старые $m,K$ в обеих правых частях. Проверяем положительную полуопределённость $K$, сумму компонент $\theta$ и изменение оценки при уменьшении шага вдвое.

In [ ]:
import _bootstrap  # noqa: F401

import time
import numpy as np
import matplotlib.pyplot as plt

from discretized_filter.config import set_config
from discretized_filter.core.filter_continuous import ContinuousFilter
from discretized_filter.core.linear_filter import build_stationary_linear_model, linear_filter_step
from discretized_filter.core.smjp import sparse_mc
from discretized_filter.utils.grids import set_seed

cfg = set_config('toy_continuous_additive')
cfg.pi = cfg.pi / (cfg.pi.sum(axis=1) * cfg.delta)[:, None]
cfg.pi_init = cfg.p0[:, None] * cfg.pi
assert np.allclose(cfg.Lambda.T @ cfg.p0, 0.0, atol=1e-12)
seeds = (20260906, 20260907, 20260908)
h_obs, h_filter, h_refined = 0.001, 0.02, 0.01
variance = cfg.C[0, 0, :, 1]
assert np.allclose(cfg.C[..., 1], variance)
A, B, H, R, mean0, K0 = build_stationary_linear_model(
    cfg.pi, cfg.M_net, cfg.delta, cfg.Lambda, cfg.p0, variance,
)

def time_grid(step):
    return np.linspace(0.0, cfg.T, int(round(cfg.T / step)) + 1)

def run_linear(path, step):
    sample = path.sample(time_grid(step))
    mean, covariance = mean0.copy(), K0.copy()
    means = [mean.copy()]
    eigenvalues = [np.linalg.eigvalsh(covariance).min()]
    for dt, increment in zip(np.diff(sample.times), sample.increments):
        mean, covariance = linear_filter_step(mean, covariance, increment, float(dt), A, B, H, R)
        assert np.isfinite(mean).all() and np.isfinite(covariance).all()
        eigenvalues.append(np.linalg.eigvalsh(covariance).min())
        means.append(mean.copy())
    means = np.array(means)
    assert min(eigenvalues) >= -1e-10
    assert np.max(abs(means[:, :cfg.N].sum(axis=1) - 1.0)) < 1e-10
    return dict(times=sample.times, theta=means[:, :cfg.N], y=means @ H.T,
                min_eigenvalue=min(eigenvalues))

def run_nonlinear(path, step):
    sample = path.sample(time_grid(step))
    filt = ContinuousFilter.from_config(cfg, ht=step)
    theta, y = filt.estimate()
    probabilities, means = [theta], [y]
    for dt, increment in zip(np.diff(sample.times), sample.increments):
        filt.update(increment, dt=float(dt))
        assert np.isfinite(filt.psi).all()
        assert np.isclose(np.sum(filt.psi * cfg.delta[:, None]), 1.0)
        theta, y = filt.estimate()
        probabilities.append(theta)
        means.append(y)
    return dict(times=sample.times, theta=np.array(probabilities), y=np.array(means))

def sample_indices(run, times):
    index = np.array([np.argmin(abs(run['times'] - t)) for t in times])
    assert np.allclose(run['times'][index], times, rtol=0, atol=1e-10)
    return index

def hidden_values(theta, y, jump_ends, times):
    index = np.minimum(np.searchsorted(jump_ends, times, side='right'), len(theta) - 1)
    return np.eye(cfg.N)[theta[index]], y[index]

In [ ]:
started = time.perf_counter()
runs = {}
mse = []
refinement = []
for seed in seeds:
    set_seed(seed)
    theta, y, jump_ends = sparse_mc(
        cfg.p0, cfg.Lambda, cfg.lam, cfg.T, cfg.get_y, cfg.y_intervals,
    )
    path = cfg.get_continuous_obs(time_grid(h_obs), theta, y, jump_ends, seed=seed + 1000)
    linear = run_linear(path, h_filter)
    linear_refined = run_linear(path, h_refined)
    nonlinear = run_nonlinear(path, h_filter)
    nonlinear_refined = run_nonlinear(path, h_refined)
    runs[seed] = dict(theta=theta, y=y, jump_ends=jump_ends,
                     linear=linear, linear_refined=linear_refined,
                     nonlinear=nonlinear, nonlinear_refined=nonlinear_refined)
    comparisons = (linear_refined, nonlinear_refined)
    # Общая сетка .02 для ошибок к истинной траектории; начальный момент исключён.
    times = linear['times'][1:]
    true_theta, true_y = hidden_values(theta, y, jump_ends, times)
    rows = []
    for run in comparisons:
        index = sample_indices(run, times)
        rows.append([np.mean(np.sum((run['theta'][index] - true_theta) ** 2, axis=1)),
                     np.mean(np.sum((run['y'][index] - true_y) ** 2, axis=1))])
    mse.append(rows)
    rows = []
    for coarse, fine in ((linear, linear_refined), (nonlinear, nonlinear_refined)):
        index = sample_indices(fine, times)
        rows.append([np.sqrt(np.mean(np.sum((coarse['theta'][1:] - fine['theta'][index]) ** 2, axis=1))),
                     np.sqrt(np.mean(np.sum((coarse['y'][1:] - fine['y'][index]) ** 2, axis=1)))])
    refinement.append(rows)

mse = np.array(mse)
refinement = np.array(refinement)
print(f'Расчёт трёх семян: {time.perf_counter() - started:.2f} с')
print('Средняя квадратичная ошибка: theta, Y (сумма координат)')
for name, values in zip(('Линейный', 'Нелинейный'), mse.mean(axis=0)):
    print(name, values)
print('Среднее RMS различие шагов .02/.01: theta, Y')
for name, values in zip(('Линейный', 'Нелинейный'), refinement.mean(axis=0)):
    print(name, values)
linear_runs = [run[key] for run in runs.values() for key in ('linear', 'linear_refined')]
print('Минимальное собственное число K:', min(run['min_eigenvalue'] for run in linear_runs))
print('Максимальное отклонение суммы theta от 1:',
      max(np.max(abs(run['theta'].sum(axis=1) - 1)) for run in linear_runs))
print('Диапазон линейных оценок theta:',
      min(run['theta'].min() for run in linear_runs), max(run['theta'].max() for run in linear_runs))

## Оценки на одной траектории

На рисунке используется первое из трёх семян и шаг $0.01$ обоих методов. Линейная оценка не обязана лежать на носителе скрытого состояния. Нелинейная оценка минимизирует среднюю квадратичную ошибку среди всех допустимых оценок в точной постановке, линейная — среди линейных. Это утверждение относится к математическому ожиданию; отдельная траектория или три семени не гарантируют превосходства на каждом интервале.

In [ ]:
example = runs[seeds[0]]
linear, nonlinear = example['linear_refined'], example['nonlinear_refined']
starts = np.r_[0.0, example['jump_ends'][:-1]]
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].step(np.r_[starts, cfg.T], np.r_[example['theta'], example['theta'][-1]],
             where='post', color='0.65', label='Истинное θ')
axes[0].plot(nonlinear['times'], nonlinear['theta'][:, 1], color='C0', label='Нелинейный')
axes[0].plot(linear['times'], linear['theta'][:, 1], '--', color='C1', label='Калман–Бьюси')
axes[0].set_ylabel('Оценка θ1')
for dim, ax in enumerate(axes[1:]):
    ax.step(np.r_[starts, cfg.T], np.r_[example['y'][:, dim], example['y'][-1, dim]],
            where='post', color='0.65', label='Истинное Y')
    ax.plot(nonlinear['times'], nonlinear['y'][:, dim], color='C0')
    ax.plot(linear['times'], linear['y'][:, dim], '--', color='C1')
    ax.set_ylabel(f'Y{dim + 1}')
for ax in axes:
    ax.grid(alpha=0.2)
axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
axes[1].legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)
axes[-1].set_xlabel('Время')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
times = linear['times']
true_theta, true_y = hidden_values(example['theta'], example['y'], example['jump_ends'], times)
for run, label, color, style in ((nonlinear, 'Нелинейный', 'C0', '-'),
                                 (linear, 'Калман–Бьюси', 'C1', '--')):
    axes[0].plot(times, np.sum((run['theta'] - true_theta) ** 2, axis=1),
                 label=label, color=color, ls=style)
    axes[1].plot(times, np.sum((run['y'] - true_y) ** 2, axis=1),
                 label=label, color=color, ls=style)
for ax, title in zip(axes, ('Ошибка θ', 'Ошибка Y')):
    ax.set(xlabel='Время', ylabel='Квадрат нормы ошибки', title=title)
    ax.grid(alpha=0.2)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.86))
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for metric, ax in enumerate(axes):
    for seed_index, values in enumerate(mse):
        ax.plot([0, 1], values[:, metric], 'o-', color='0.65', alpha=0.55,
                label='Отдельное семя' if seed_index == 0 else None)
    ax.plot([0, 1], mse.mean(axis=0)[:, metric], 'o-', color='C0', lw=2, label='Среднее трёх семян')
    ax.set_xticks([0, 1], ['Калман–Бьюси', 'Нелинейный'])
    ax.set_ylabel(('Средний квадрат ошибки θ', 'Средний квадрат ошибки Y')[metric])
    ax.set_xlim(-0.15, 1.15)
    ax.grid(axis='y', alpha=0.2)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.85))
plt.show()

Ковариация $K$ в линейном фильтре — безусловная ковариация ошибки линейной оценки. Она не задаёт гауссовскую условную плотность $Y$ в этой негауссовской модели. Поэтому плотности не восстанавливаются из $m,K$: снимки нелинейной плотности и её сравнение с дискретным фильтром находятся в [ноутбуке сходимости](continuous_filter_convergence.ipynb).